# Patch newline-key values

Remove newline-key values while keeping the attention pattern fixed.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    if (candidate / "results/line_breaks_results").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/06_layer1_mechanisms"))
from common import *
EXAMPLE_SEED=6001
ctx=load_context(EXAMPLE_SEED)
model,vocab,batch,cfg,device=ctx.model,ctx.vocab,ctx.batch,ctx.cfg,ctx.device
tokens=batch.tokens[0].detach().cpu().numpy()
print("checkpoint",ctx.checkpoint["step"],"seed",EXAMPLE_SEED,"line width",int(batch.line_lengths[0]))
print("newline positions",newline_positions(tokens,vocab))

/Users/rohanbuluswar/Desktop/Line Break Project/.venv_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building vocab...
('BOS', '_NEWLINE_')
checkpoint 10000 seed 6001 line width 89
newline positions [22, 44, 66, 87, 106, 129, 149]


In [2]:
V,POST="blocks.1.attn.hook_v","blocks.1.hook_resid_post"
_,c=model.run_with_cache(batch.tokens,prepend_bos=False,names_filter=[POST]); clean=c[POST][0].detach().cpu().numpy(); nls=set(newline_positions(tokens,vocab)); captured={}
def patch(x,hook):
 y=x.clone()
 for p in nls:y[:,p,1,:]=0
 return y
def cap(x,hook):captured["post"]=x.detach().cpu();return x
model.run_with_hooks(batch.tokens,prepend_bos=False,fwd_hooks=[(V,patch),(POST,cap)])
patched=captured["post"][0].numpy(); print("post-layer-1 MSE",float(np.mean((patched-clean)**2)))
plot_trajectory(shared_pca([clean,patched]),["clean","newline values removed"],"Newline-key value patch","figures/layer1_newline_value_patch.png")

post-layer-1 MSE 20.221221923828125
